In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import display

from ncpu_simplified import (
    ExperimentConfig, GeometryConfig, InterleavedLayout, ModelConfig,
    TrainingConfig, evaluate_widths, format_results, load_model, train_seeds, validate,
)
from ncpu_simplified.visualize import notebook_viewer, save_gif

ROOT = Path.cwd().parent if Path.cwd().name == 'run' else Path.cwd()
CHECKPOINT_DIR = ROOT / 'checkpoints'
RESUME = False
SEEDS = (0,)
WANDB_PROJECT = None  # Example: 'ncpu-simplified'; requires the optional tracking extra.
WANDB_GROUP = 'canonical-4bit'

config = ExperimentConfig(
    geometry=GeometryConfig(
        bits=4, sx=3, sy=3,
        border_left=3, border_right=3, border_top=3, border_bottom=3,
    ),
    model=ModelConfig(
        channels=3,
        hidden_size=57,
        fixed_kernels=('identity', 'sobel_x', 'sobel_y'),
        fixed_laplacian=False,
        learnable_kernels=1,
        learnable_kernel_init='laplacian',
        gate='none',
        gate_bias=1.0,
        fire_rate=1.0,  # 1.0 is synchronous; use (0, 1) for asynchronous updates.
        input_channel=0,
        input_mode='mutable',
        padding='zeros',
        max_abs_state=10.0,
    ),
    training=TrainingConfig(
        updates=1500,
        batch_size=128,
        free_steps=70,
        supervision_steps=70,
        learning_rate=1e-3,
        final_learning_rate=3e-4,
        warmup_updates=0,
        weight_decay=0.0,
        grad_clip=1.0,
        seed=0,
        validation_every=50,
        checkpoint_every=50,
        device='auto',
    ),
)

report = validate(config)
print(report)
report.require_success()

In [ ]:
seed_results = train_seeds(
    config,
    SEEDS,
    CHECKPOINT_DIR,
    resume=RESUME,
    progress_every=25,
    wandb_project=WANDB_PROJECT,
    wandb_group=WANDB_GROUP,
)
for result in seed_results:
    print(f'seed {result.seed}: best validation MSE={result.best_validation_loss:.6g}')
best_seed = min(seed_results, key=lambda result: result.best_validation_loss)
history = best_seed.history
model, trained_config, _ = load_model(CHECKPOINT_DIR / 'best.pt', config.training.device)

steps = [row['update'] for row in history]
losses = [row['loss'] for row in history]
validation_points = [(row['update'], row['validation_loss']) for row in history if row['validation_loss'] is not None]
plt.figure(figsize=(8, 4))
plt.plot(steps, losses, linewidth=1, label='training')
if validation_points:
    plt.plot(*zip(*validation_points), marker='o', markersize=3, label='exhaustive validation')
plt.yscale('log')
plt.xlabel('update')
plt.ylabel('masked MSE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# None evaluates every ordered pair: 256 at 4 bits, 4096 at 6, 65536 at 8.
VALIDATION_MAX_PAIRS = None
results = evaluate_widths(
    model,
    trained_config,
    widths=(4, 6, 8),
    batch_size=256,
    max_pairs=VALIDATION_MAX_PAIRS,
    seed=0,
)
print(format_results(results))

plt.figure(figsize=(8, 4))
for result in results:
    timeline = range(result.step_start, result.step_end + 1)
    plt.plot(timeline, result.exact_by_step[result.step_start:result.step_end + 1], label=f'{result.bits} bits')
plt.xlabel('evolution step')
plt.ylabel('exact accuracy')
plt.ylim(0, 1.01)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
INFERENCE_BITS = 8
OPERAND_A = 173
OPERAND_B = 82
INFERENCE_STEPS = 200

inference_layout = InterleavedLayout(replace(trained_config.geometry, bits=INFERENCE_BITS))
inputs, _ = inference_layout.render_batch(torch.tensor([OPERAND_A]), torch.tensor([OPERAND_B]))
with torch.no_grad():
    rollout = model(model.initial_state(inputs.to(model.device)), INFERENCE_STEPS)[0].cpu()
decoded = inference_layout.decode_output(rollout[:, trained_config.model.input_channel])
gif_path = save_gif(
    rollout, ROOT / 'run' / 'inference.gif', duration_ms=70, scale=24,
    layout=inference_layout, config=trained_config, operand_a=OPERAND_A, operand_b=OPERAND_B,
)
display(notebook_viewer(
    rollout, inference_layout, trained_config, OPERAND_A, OPERAND_B, duration_ms=70,
))